In [1]:
import pandas as pd
from pathlib import Path

# Correct path (it's inside 'annotated')
base_path = Path("/home/pv_linux/linux_Projects/Terminology/ACTER-master/en/htfl")
iob_path = base_path / "annotated" / "annotations" / "sequential_annotations" / "iob_annotations" / "without_named_entities"

all_data = []

# Get TSV files (skip Zone.Identifier files)
tsv_files = [f for f in iob_path.glob("*.tsv") if not f.name.endswith(':Zone.Identifier')]
print(f"Found {len(tsv_files)} files to process")

for tsv_file in tsv_files:
    print(f"Processing: {tsv_file.name}")
    
    with open(tsv_file, 'r', encoding='utf-8') as f:
        sentence_tokens = []
        current_term = []
        terms_in_sentence = []
        
        for line in f:
            line = line.strip()
            
            if not line:  # Empty line = end of sentence
                if sentence_tokens:
                    # Save any remaining term
                    if current_term:
                        terms_in_sentence.append(' '.join(current_term))
                        current_term = []
                    
                    # Create the row
                    sentence = ' '.join(sentence_tokens)
                    annotations = ', '.join(terms_in_sentence) if terms_in_sentence else ''
                    
                    all_data.append({
                        'sentence': sentence,
                        'annotations': annotations
                    })
                    
                    # Reset
                    sentence_tokens = []
                    terms_in_sentence = []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    token = parts[0]
                    label = parts[1]
                    
                    # Add token to sentence
                    sentence_tokens.append(token)
                    
                    # Handle terms
                    if label == 'B':  # Beginning of term
                        if current_term:  # Save previous term
                            terms_in_sentence.append(' '.join(current_term))
                        current_term = [token]
                    elif label == 'I':  # Inside term
                        current_term.append(token)
                    else:  # 'O' - outside term
                        if current_term:  # Save current term
                            terms_in_sentence.append(' '.join(current_term))
                            current_term = []
        
        # Don't forget last sentence in file
        if sentence_tokens:
            if current_term:
                terms_in_sentence.append(' '.join(current_term))
            sentence = ' '.join(sentence_tokens)
            annotations = ', '.join(terms_in_sentence) if terms_in_sentence else ''
            all_data.append({
                'sentence': sentence,
                'annotations': annotations
            })

# Create DataFrame
df = pd.DataFrame(all_data)

print(f"\n{'='*60}")
print(f"DataFrame Created Successfully!")
print(f"{'='*60}")
print(f"Total sentences: {len(df)}")
print(f"\nFirst 10 rows:")
print(df.head(10))

# Show sample
print(f"\nSample sentences with terms:")
print(df[df['annotations'] != ''].head(5))

# Display
df

Found 190 files to process
Processing: htfl_en_079_seq_terms.tsv
Processing: htfl_en_072_seq_terms.tsv
Processing: htfl_en_076_seq_terms.tsv
Processing: htfl_en_025_seq_terms.tsv
Processing: htfl_en_073_seq_terms.tsv
Processing: htfl_en_007_seq_terms.tsv
Processing: htfl_en_110_seq_terms.tsv
Processing: htfl_en_177_seq_terms.tsv
Processing: htfl_en_139_seq_terms.tsv
Processing: htfl_en_063_seq_terms.tsv
Processing: htfl_en_052_seq_terms.tsv
Processing: htfl_en_036_seq_terms.tsv
Processing: htfl_en_035_seq_terms.tsv
Processing: htfl_en_137_seq_terms.tsv
Processing: htfl_en_026_seq_terms.tsv
Processing: htfl_en_049_seq_terms.tsv
Processing: htfl_en_119_seq_terms.tsv
Processing: htfl_en_061_seq_terms.tsv
Processing: htfl_en_015_seq_terms.tsv
Processing: htfl_en_005_seq_terms.tsv
Processing: htfl_en_047_seq_terms.tsv
Processing: htfl_en_113_seq_terms.tsv
Processing: htfl_en_057_seq_terms.tsv
Processing: htfl_en_178_seq_terms.tsv
Processing: htfl_en_059_seq_terms.tsv
Processing: htfl_en_078

,sentence,annotations
0,Nicorandil ameliorates mitochondrial dysfuncti...,"Nicorandil, mitochondrial dysfunction, doxorub..."
1,"Despite of its known cardiotoxicity , doxorubi...","cardiotoxicity, doxorubicin, anti-neoplastic a..."
2,"In the present study , the cardioprotective ef...","cardioprotective, nicorandil, hemodynamic, mit..."
3,Doxorubicin was injected i.p. over 2 weeks to ...,"Doxorubicin, injected"
4,Nicorandil ( 3 mg / kg / day ) was given orall...,"Nicorandil, orally, doxorubicin"
...,...,...
2427,"At study closure , the 809 patients who had un...","patients, randomization"
2428,The primary outcome occurred in 116 of 404 pat...,"primary outcome, patients, CRT, hazard ratio, ..."
2429,There were 45 deaths in the CRT group and 26 i...,"deaths, CRT, hazard ratio, CI, P=0.02"
2430,CONCLUSIONS :,


In [2]:
# Save to Excel
df.to_excel('htfl_data.xlsx', index=False)
print("Saved to htfl_data.xlsx")

Saved to htfl_data.xlsx


In [3]:
# Keep only sentences that have annotations
df_clean = df[df['annotations'] != '']
print(f"Kept {len(df_clean)} sentences with terms")
df_clean.to_excel('htfl_data_with_terms.xlsx', index=False)

Kept 1989 sentences with terms


In [ ]:
# Remove rows with just section headers
df_clean = df[df['annotations'] != ''].copy()  # Keep only rows with annotations
print(f"Original rows: {len(df)}")
print(f"After cleaning: {len(df_clean)}")
df_clean.to_excel('htfl_data_clean.xlsx', index=False)
```

---

## 2️⃣ **How the mapping works:**

For the text file:
```
/htfl/annotated/texts/htfl_en_001.txt
```

The annotations are in the **corresponding TSV file**:
```
/htfl/annotated/annotations/sequential_annotations/iob_annotations/without_named_entities/htfl_en_001_seq_terms.tsv

SyntaxError: invalid syntax (1698977376.py, line 6)

In [5]:
from pathlib import Path

base_path = Path("/home/pv_linux/linux_Projects/Terminology/ACTER-master/en/htfl")

# The text file
text_file = base_path / "annotated" / "texts" / "htfl_en_001.txt"
print(f"Text file exists: {text_file.exists()}")

# The corresponding annotation file
ann_file = base_path / "annotated" / "annotations" / "sequential_annotations" / "iob_annotations" / "without_named_entities" / "htfl_en_001_seq_terms.tsv"
print(f"Annotation file exists: {ann_file.exists()}")

# Show first 30 lines of the annotation file
if ann_file.exists():
    print("\nFirst 30 lines of annotation file:")
    print("="*60)
    with open(ann_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < 30:
                print(line.rstrip())
            else:
                break

Text file exists: True
Annotation file exists: True

First 30 lines of annotation file:
National	O
trends	O
in	O
patient	B
safety	O
for	O
four	O
common	O
conditions	B
,	O
2005	O
-	O
2011	O
.	O

The	O
effects	O
of	O
more	O
than	O
a	O
decade	O
of	O
national	O
efforts	O
dedicated	O
to	O
improve	O
patient	B
safety	O


In [6]:
# Keep only rows that have annotations (remove empty annotation rows)
df_clean = df[df['annotations'] != ''].copy()

print(f"Before: {len(df)} rows")
print(f"After: {len(df_clean)} rows")
print(f"Removed: {len(df) - len(df_clean)} empty rows")

# Save cleaned version
df_clean.to_excel('htfl_data_clean.xlsx', index=False)
print("\nSaved to htfl_data_clean.xlsx ✓")

Before: 2432 rows
After: 1989 rows
Removed: 443 empty rows

Saved to htfl_data_clean.xlsx ✓


In [7]:
import pandas as pd
from pathlib import Path

# Paths
base_path = Path("/home/pv_linux/linux_Projects/Terminology/ACTER-master/en/htfl/annotated")
texts_dir = base_path / "texts"
annotations_dir = base_path / "annotations" / "sequential_annotations" / "iob_annotations" / "without_named_entities"

data = []

# Get all text files
text_files = sorted([f for f in texts_dir.glob("*.txt") if not f.name.endswith(':Zone.Identifier')])
print(f"Found {len(text_files)} files\n")

for text_file in text_files:
    file_id = text_file.stem
    
    # Read full text
    with open(text_file, 'r', encoding='utf-8') as f:
        full_text = f.read().strip()
    
    # Read annotations
    ann_file = annotations_dir / f"{file_id}_seq_terms.tsv"
    if not ann_file.exists():
        continue
    
    # Extract all terms
    terms = []
    current_term = []
    
    with open(ann_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if current_term:
                    terms.append(' '.join(current_term))
                    current_term = []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    token, label = parts[0], parts[1]
                    
                    if label == 'B':
                        if current_term:
                            terms.append(' '.join(current_term))
                        current_term = [token]
                    elif label == 'I':
                        current_term.append(token)
                    else:
                        if current_term:
                            terms.append(' '.join(current_term))
                            current_term = []
    
    if current_term:
        terms.append(' '.join(current_term))
    
    # Remove duplicates, keep order
    unique_terms = list(dict.fromkeys(terms))
    annotations = ', '.join(unique_terms)
    
    data.append({
        'sentence_id': file_id,
        'sentence': full_text,
        'annotations': annotations
    })
    
    print(f"✓ {file_id}: {len(unique_terms)} terms")

# Create DataFrame
df = pd.DataFrame(data)

print(f"\n{'='*60}")
print(f"Total documents: {len(df)}")
print(f"First document: {df.iloc[0]['sentence_id']}")
print(f"Text length: {len(df.iloc[0]['sentence'])} chars")

# Save
df.to_excel('htfl_documents.xlsx', index=False)
print("✓ Saved to htfl_documents.xlsx")

df

Found 190 files

✓ htfl_en_001: 16 terms
✓ htfl_en_002: 46 terms
✓ htfl_en_003: 26 terms
✓ htfl_en_004: 28 terms
✓ htfl_en_005: 28 terms
✓ htfl_en_006: 20 terms
✓ htfl_en_007: 7 terms
✓ htfl_en_008: 33 terms
✓ htfl_en_009: 28 terms
✓ htfl_en_010: 6 terms
✓ htfl_en_011: 40 terms
✓ htfl_en_012: 34 terms
✓ htfl_en_013: 26 terms
✓ htfl_en_014: 34 terms
✓ htfl_en_015: 23 terms
✓ htfl_en_016: 35 terms
✓ htfl_en_017: 30 terms
✓ htfl_en_018: 17 terms
✓ htfl_en_019: 49 terms
✓ htfl_en_020: 32 terms
✓ htfl_en_021: 15 terms
✓ htfl_en_022: 10 terms
✓ htfl_en_023: 21 terms
✓ htfl_en_024: 31 terms
✓ htfl_en_025: 34 terms
✓ htfl_en_026: 39 terms
✓ htfl_en_027: 23 terms
✓ htfl_en_028: 31 terms
✓ htfl_en_029: 30 terms
✓ htfl_en_030: 29 terms
✓ htfl_en_031: 21 terms
✓ htfl_en_032: 47 terms
✓ htfl_en_033: 84 terms
✓ htfl_en_034: 29 terms
✓ htfl_en_035: 33 terms
✓ htfl_en_036: 36 terms
✓ htfl_en_037: 31 terms
✓ htfl_en_038: 50 terms
✓ htfl_en_039: 43 terms
✓ htfl_en_040: 26 terms
✓ htfl_en_041: 45 terms
✓

,sentence_id,sentence,annotations
0,htfl_en_001,National trends in patient safety for four com...,"patient, conditions, Patient, adverse event, p..."
1,htfl_en_002,Cross-talk between the heart and adipose tissu...,"Cross-talk, heart, adipose tissue, cachectic h..."
2,htfl_en_003,Diagnosis of heart failure with preserved ejec...,"Diagnosis, heart failure with preserved ejecti..."
3,htfl_en_004,Treatment of anemia in patients with heart dis...,"anemia, patients, heart disease, clinical prac..."
4,htfl_en_005,Treatment of anemia in patients with heart dis...,"anemia, patients, heart disease, clinical, blo..."
...,...,...,...
185,htfl_en_186,Study PARADIGM-HF - a paradigm shift in the tr...,"PARADIGM-HF, chronic heart failure, Chronic he..."
186,htfl_en_187,Effect of Caloric Restriction or Aerobic Exerc...,"Peak Oxygen Consumption, Quality of Life, Obes..."
187,htfl_en_188,Fully Magnetically Levitated Left Ventricular ...,Fully Magnetically Levitated Left Ventricular ...
188,htfl_en_189,Tools for Economic Analysis of Patient Managem...,"Patient, Heart Failure, disease, heart failure..."


In [8]:
import pandas as pd
from pathlib import Path
import re

# Paths
base_path = Path("/home/pv_linux/linux_Projects/Terminology/ACTER-master/en/htfl/annotated")
texts_dir = base_path / "texts"
annotations_dir = base_path / "annotations" / "sequential_annotations" / "iob_annotations" / "without_named_entities"

data = []

# Get all text files
text_files = sorted([f for f in texts_dir.glob("*.txt") if not f.name.endswith(':Zone.Identifier')])
print(f"Found {len(text_files)} files\n")

for text_file in text_files:
    file_id = text_file.stem
    
    # Read full text
    with open(text_file, 'r', encoding='utf-8') as f:
        full_text = f.read().strip()
    
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', full_text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    # Read annotations and extract all terms
    ann_file = annotations_dir / f"{file_id}_seq_terms.tsv"
    if not ann_file.exists():
        continue
    
    all_terms = []
    current_term = []
    
    with open(ann_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            
            if not line:
                if current_term:
                    all_terms.append(' '.join(current_term))
                    current_term = []
            else:
                parts = line.split('\t')
                if len(parts) >= 2:
                    token, label = parts[0], parts[1]
                    
                    if label == 'B':
                        if current_term:
                            all_terms.append(' '.join(current_term))
                        current_term = [token]
                    elif label == 'I':
                        current_term.append(token)
                    else:
                        if current_term:
                            all_terms.append(' '.join(current_term))
                            current_term = []
    
    if current_term:
        all_terms.append(' '.join(current_term))
    
    # Match sentences with their terms
    for sent_num, sentence in enumerate(sentences, 1):
        sentence_terms = []
        for term in all_terms:
            if term.lower() in sentence.lower():
                sentence_terms.append(term)
        
        unique_terms = list(dict.fromkeys(sentence_terms))
        annotations = ', '.join(unique_terms) if unique_terms else ''
        
        sentence_id = f"{file_id}_s{sent_num:03d}"
        
        data.append({
            'sentence_id': sentence_id,
            'sentence': sentence,
            'annotations': annotations
        })
    
    print(f"✓ {file_id}: {len(sentences)} sentences")

# Create DataFrame
df = pd.DataFrame(data)

print(f"\n{'='*60}")
print(f"Total sentences: {len(df)}")
print(f"With terms: {len(df[df['annotations'] != ''])}")
print(f"Without terms: {len(df[df['annotations'] == ''])}")

# Save both versions
df.to_excel('htfl_sentences_all.xlsx', index=False)
print(f"✓ Saved ALL sentences")

df_with_terms = df[df['annotations'] != '']
df_with_terms.to_excel('htfl_sentences_with_terms.xlsx', index=False)
print(f"✓ Saved ONLY sentences with terms ({len(df_with_terms)} rows)")

df

Found 190 files

✓ htfl_en_001: 8 sentences
✓ htfl_en_002: 13 sentences
✓ htfl_en_003: 8 sentences
✓ htfl_en_004: 10 sentences
✓ htfl_en_005: 16 sentences
✓ htfl_en_006: 3 sentences
✓ htfl_en_007: 3 sentences
✓ htfl_en_008: 12 sentences
✓ htfl_en_009: 13 sentences
✓ htfl_en_010: 4 sentences
✓ htfl_en_011: 13 sentences
✓ htfl_en_012: 9 sentences
✓ htfl_en_013: 9 sentences
✓ htfl_en_014: 15 sentences
✓ htfl_en_015: 12 sentences
✓ htfl_en_016: 18 sentences
✓ htfl_en_017: 10 sentences
✓ htfl_en_018: 7 sentences
✓ htfl_en_019: 12 sentences
✓ htfl_en_020: 12 sentences
✓ htfl_en_021: 6 sentences
✓ htfl_en_022: 4 sentences
✓ htfl_en_023: 15 sentences
✓ htfl_en_024: 12 sentences
✓ htfl_en_025: 10 sentences
✓ htfl_en_026: 17 sentences
✓ htfl_en_027: 11 sentences
✓ htfl_en_028: 11 sentences
✓ htfl_en_029: 9 sentences
✓ htfl_en_030: 13 sentences
✓ htfl_en_031: 12 sentences
✓ htfl_en_032: 15 sentences
✓ htfl_en_033: 49 sentences
✓ htfl_en_034: 9 sentences
✓ htfl_en_035: 12 sentences
✓ htfl_en_036: 

,sentence_id,sentence,annotations
0,htfl_en_001_s001,National trends in patient safety for four com...,"patient, conditions, Patient"
1,htfl_en_001_s002,The effects of more than a decade of national ...,"patient, Patient"
2,htfl_en_001_s003,This study used the Medicare Patient Safety Mo...,"patient, conditions, Patient, adverse event, p..."
3,htfl_en_001_s004,The analysis included a large study sample wit...,"patient, Patient, patients, hospitals"
4,htfl_en_001_s005,The results show a significant decline in adve...,"adverse event, acute myocardial infarction, co..."
...,...,...,...
2055,htfl_en_190_s002,Heart failure (HF) is a debilitating chronic d...,"Heart Failure, Heart failure, HF, chronic, dis..."
2056,htfl_en_190_s003,Nurses in all settings have an essential role ...,"disease, Nurses, patients, nurses"
2057,htfl_en_190_s004,This article describes the pathophysiology of ...,"Pathophysiology, Diagnosis, Nursing, HF, patho..."
2058,htfl_en_190_s005,It is crucial for nurses to understand the pat...,"Pathophysiology, Nursing, HF, Nurses, pathophy..."


In [9]:
# After creating the DataFrame, add this cleaning step:

def clean_annotations(annotation_str):
    """Remove meaningless single-letter terms and clean up"""
    if not annotation_str:
        return ''
    
    terms = [t.strip() for t in annotation_str.split(',')]
    
    # Filter out problematic terms
    cleaned_terms = []
    for term in terms:
        # Skip single letters (P, I, CI, etc.)
        if len(term) <= 1:
            continue
        # Skip very short terms that are just punctuation or numbers
        if len(term) <= 2 and not term.isalpha():
            continue
        # Keep the term
        cleaned_terms.append(term)
    
    # Remove duplicates, keep order
    unique_terms = list(dict.fromkeys(cleaned_terms))
    return ', '.join(unique_terms)

# Apply cleaning
df['annotations'] = df['annotations'].apply(clean_annotations)

print("✓ Cleaned annotations")

# Recount
print(f"After cleaning:")
print(f"Sentences with terms: {len(df[df['annotations'] != ''])}")
print(f"Sentences without terms: {len(df[df['annotations'] == ''])}")

# Save cleaned versions
df.to_excel('htfl_sentences_all_cleaned.xlsx', index=False)
df_with_terms = df[df['annotations'] != '']
df_with_terms.to_excel('htfl_sentences_with_terms_cleaned.xlsx', index=False)

print("✓ Saved cleaned files")
df

✓ Cleaned annotations
After cleaning:
Sentences with terms: 1996
Sentences without terms: 64
✓ Saved cleaned files


,sentence_id,sentence,annotations
0,htfl_en_001_s001,National trends in patient safety for four com...,"patient, conditions, Patient"
1,htfl_en_001_s002,The effects of more than a decade of national ...,"patient, Patient"
2,htfl_en_001_s003,This study used the Medicare Patient Safety Mo...,"patient, conditions, Patient, adverse event, p..."
3,htfl_en_001_s004,The analysis included a large study sample wit...,"patient, Patient, patients, hospitals"
4,htfl_en_001_s005,The results show a significant decline in adve...,"adverse event, acute myocardial infarction, co..."
...,...,...,...
2055,htfl_en_190_s002,Heart failure (HF) is a debilitating chronic d...,"Heart Failure, Heart failure, HF, chronic, dis..."
2056,htfl_en_190_s003,Nurses in all settings have an essential role ...,"disease, Nurses, patients, nurses"
2057,htfl_en_190_s004,This article describes the pathophysiology of ...,"Pathophysiology, Diagnosis, Nursing, HF, patho..."
2058,htfl_en_190_s005,It is crucial for nurses to understand the pat...,"Pathophysiology, Nursing, HF, Nurses, pathophy..."
